# Workflows: how many magnitude 5 earthquakes for each magnitude 6?

## Learning objectives

- Turn a README's "run these in this order" into a declared workflow, where each step names what it reads and what it writes.
- Let the tool decide what's stale, so a change reruns only the steps after it.
- Declare the parameters and the code as inputs, not only the data.
- Record a decision, the magnitude where the catalog is complete, in the parameters and the commit message.
- Delete every output and get them all back from the raw data with one command.

## The problem

- `project` computes one number from the 2024 earthquake catalog: the Gutenberg-Richter b-value. Each step down in magnitude multiplies the count of earthquakes by 10^b. With b near 1, there are about ten magnitude 5 earthquakes for each magnitude 6.
- The README says to run three scripts in order, `subset.py`, `count.py` and `fit.py`, and after a change to `params.yaml` to "rerun the steps that need it".
- The last commit limited the catalog to shallow earthquakes and reran only `fit.py`. The committed number comes from the whole catalog, and nothing says so.
- The number is wrong a second way. The catalog misses small earthquakes, and the fit starts at magnitude 4.0, where the catalog isn't complete.

## What you'll do

1. Step 1: write a Snakefile that declares the three steps, build from the raw data, and commit the fresh result. This is the short path, about 15 minutes, and it has its own check.
2. Step 2, optional: declare the parameters and the scripts too, then choose the start magnitude and record why.
3. Step 3, optional: declare the step the README skipped, then delete every output and rebuild.

- Steps 2 and 3 take about 20 more minutes.
- Only reading? Each step shows the failure and how to approach the fix.

## The rule

- The document and the data are the pets. Everything that runs them is cattle.
- Here the pets are the catalog, the scripts, the parameters and the Snakefile, all in Git. The cattle are `results/` and `.snakemake/`. Delete them, and the workflow builds them again.
- `metrics.json` is committed, so a reader sees the number without running anything. It has to come back byte for byte from the raw data.

## Get a terminal

- You read this page and paste commands into a terminal. Nothing runs on the page.
- The output shown here is recorded from the untouched project, so it stops at the failures. The fixes are yours.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-02-workflows?urlpath=lab/workspaces/exercise)**

- Free, with nothing to install. The first start can take a few minutes.
- The terminal is on the left, already in the `project` folder. This page is on the right.
- Copy a command: select it and press Ctrl+C, or Cmd+C on a Mac.
- Paste it in the terminal: Ctrl+Shift+V, or right-click and choose Paste. Cmd+V on a Mac is untested, so right-click if it doesn't work.
- Edit a file: double-click it in the file panel on the far left. Save with Ctrl+S.
- A session is temporary and has 2 GB of memory. Download anything you want to keep.

**Or use your own terminal**

- Optional: [use uv on your own machine](../../uv.md).

- Linux or macOS, on x86_64 or ARM, or Windows through WSL2. On Windows, keep the folder in your WSL2 home, not under `/mnt/c`.
- You need [conda](https://github.com/conda-forge/miniforge). You don't need Jupyter.
- Paste this once:

```bash
git clone --branch binder-02-workflows --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git workflows
cd workflows
conda env create --quiet --file .binder/environment.yml
conda activate repro-workflows
python scripts/setup_project.py
cd project
```

- Every command below runs in that `project` folder.
- To start over, go to the folder above it and run `python scripts/setup_project.py --reset`. This deletes your work.

## The project

- `project` has the 2024 catalog, three scripts, `params.yaml` and a committed `metrics.json`, in two commits.
- The README is the only record of which step reads what, and in which order.

```bash
cat README.md
git log --format='%h  %s'
cat params.yaml
cat metrics.json
```

In [1]:
# Recorded output. You don't need to run this.
import sys

sys.path.insert(0, ".exercise")
from page import run

run("cat README.md", "git log --format='%h  %s'", "cat params.yaml", "cat metrics.json")

$ cat README.md
# How many magnitude 5 earthquakes for each magnitude 6?

We report the Gutenberg-Richter b-value of the 2024 catalog. Each step down in magnitude multiplies the number of earthquakes by 10^b.

Run these in this order:

1. `python subset.py` keeps the earthquakes `params.yaml` asks for and writes `results/subset.csv`.
2. `python count.py` counts them per 0.1 of magnitude and writes `results/counts.csv`.
3. `python fit.py` fits the b-value and writes `metrics.json`.

After a change to `params.yaml`, rerun the steps that need it.

Data: `data/earthquakes.csv`, from the USGS ANSS Comprehensive Earthquake Catalog.

$ git log --format='%h  %s'
c8a0c62  Keep only shallow earthquakes, 70 km or less. Reran the fit, and the b-value doesn't change
4e8ac64  Add the 2024 catalog, the three scripts and the first fit

$ cat params.yaml
# Which earthquakes subset.py keeps.
min_magnitude: 4.0
max_depth_km: 70

$ cat metrics.json
{
  "b_value": 0.802,
  "b_uncertainty": 0.007,
  "quakes

## Step 1: declare the steps

**Problem:** the README gives an order. Run all three scripts in that order, and the committed number moves.

```bash
python subset.py && python count.py && python fit.py
git diff metrics.json
```

In [2]:
# Recorded output. You don't need to run this.
run("python subset.py && python count.py && python fit.py", "git diff metrics.json")

$ python subset.py && python count.py && python fit.py
subset: kept 9496 of 14175 earthquakes
count: 36 magnitude bins, 9496 earthquakes
{
  "b_value": 0.744,
  "b_uncertainty": 0.008,
  "quakes_per_unit_magnitude": 5.5,
  "n_earthquakes": 9496,
  "lowest_magnitude": 4.0
}

$ git diff metrics.json
diff --git a/metrics.json b/metrics.json
index 9fe5fac..443aae2 100644
--- a/metrics.json
+++ b/metrics.json
@@ -1,7 +1,7 @@
 {
-  "b_value": 0.802,
-  "b_uncertainty": 0.007,
-  "quakes_per_unit_magnitude": 6.3,
-  "n_earthquakes": 14175,
+  "b_value": 0.744,
+  "b_uncertainty": 0.008,
+  "quakes_per_unit_magnitude": 5.5,
+  "n_earthquakes": 9496,
   "lowest_magnitude": 4.0
 }


**Facts**

- The last commit set `max_depth_km` to 70 and reran only `fit.py`. `fit.py` reads `results/counts.csv`, and `count.py` had written that from the whole catalog, so the "unchanged" b-value in that commit never saw the new depth limit.
- A README order is a procedure. It relies on a person to know what's stale, and the person was wrong.
- A Snakefile is a declaration. Each rule names its input files, its output files and the command. Snakemake builds the graph of steps from the file names, and it reruns a rule when an input is newer than its output or an output is missing.
- The first rule is the default target. A rule named `all`, with the final files as its inputs, is the convention.
- `snakemake -n` is a dry run. It lists what would run and runs nothing. `snakemake --cores 1` runs it.
- `--forceall` runs every rule, whatever the file dates say. Use it once, when you adopt a workflow, because nobody can vouch for the files already there.
- `results/` and `.snakemake/` are in `.gitignore`. `metrics.json` is not.

**Steps**

1. Create a file named `Snakefile` in the `project` folder with this, and add a rule for `count.py` and a rule for `fit.py` in the same form. `count.py` reads `results/subset.csv` and writes `results/counts.csv`. `fit.py` reads `results/counts.csv` and writes `metrics.json`. Indent with four spaces.

```text
rule all:
    input: "metrics.json"

rule subset:
    input: "data/earthquakes.csv"
    output: "results/subset.csv"
    shell: "python subset.py"
```

2. Ask Snakemake what it would do. It should say there's nothing to do, and that three rules have no record of how their outputs were made. If it reports an error instead, read the line number it names.

```bash
snakemake -n
```

3. Build everything from the raw data once, and look at the result. The diff is the one the procedure gave, and the dry run now says nothing to do, with no missing records.

```bash
snakemake --cores 1 --forceall
git diff metrics.json
snakemake -n
```

4. Commit the Snakefile and the fresh result.

```bash
git add Snakefile metrics.json
git commit -m "Declare the three steps, and rebuild the result from the raw data"
```

### Check the short path

- Run this when step 1 is done. It clones your last commit into an empty folder, deletes every output and builds them with your Snakefile.
- Each line says what to do next.
- When it passes, one command rebuilds your result from the raw data. That's the end of the short path.

```bash
python ../check.py --short
```

On the starter project it fails like this:

In [3]:
# Recorded output. You don't need to run this.
run("python ../check.py --short")

$ python ../check.py --short
Note: you have uncommitted changes. This checks your last commit only.

FAIL  No Snakefile is committed. Write the rules in step 1, then git add Snakefile and commit.

0 of 1 parts pass.


## Step 2 (optional): declare the parameters and the code

**Problem**

- The rules declare the data files. `subset.py` also reads `params.yaml`, and each rule runs a script. Snakemake knows neither.
- This script changes `max_depth_km` in `params.yaml`, asks Snakemake what it would rerun, and puts the parameter back.

```bash
python ../scripts/try_params.py
```

In [4]:
# Recorded output. You don't need to run this.
run("python ../scripts/try_params.py")

$ python ../scripts/try_params.py
I changed max_depth_km in params.yaml. A different depth limit means a different selection. snakemake -n says:

Error: No Snakefile found, tried Snakefile, snakefile, workflow/Snakefile, workflow/snakefile.

I put params.yaml back.


**Facts**

- The starter project has no Snakefile yet, so the script shows that error. With the step 1 Snakefile, it prints `Nothing to be done (all requested files are present and up to date).` A changed parameter, or an edited script, leaves the result in place, and Snakemake calls it up to date.
- A step reads more than its data. Declare the parameters file and the script as inputs, and a change to either reruns the step and every step after it.
- Snakemake also has `configfile:`, `params:` and `script:` directives for this, with finer control. The input list is enough here. The [Snakemake documentation on rules](https://snakemake.readthedocs.io/en/stable/snakefiles/rules.html) has the rest.
- Below the completeness magnitude the catalog misses earthquakes, so the fit sees too few small ones and b comes out too low.
- `completeness.py` fits b from each start magnitude, 4.0 to 5.5. b rises while the catalog is incomplete, then levels off within its uncertainty.
- Choosing the start magnitude is a decision. Pick the lowest one on the level part. A higher one throws away earthquakes and widens the uncertainty for nothing. No tool makes it for you.

**Steps**

1. In the Snakefile, add `"params.yaml"` and `"subset.py"` to the inputs of `subset`, `"count.py"` to the inputs of `count`, and `"fit.py"` to the inputs of `fit`. Inputs are separated by commas: `input: "data/earthquakes.csv", "params.yaml", "subset.py"`.
2. Run the script again. It should list all three jobs.

```bash
python ../scripts/try_params.py
```

3. Read the table, and pick a start magnitude you can defend.

```bash
python completeness.py
```

In [5]:
# Recorded output. You don't need to run this.
run("python completeness.py")

$ python completeness.py
| start magnitude | earthquakes | b-value | uncertainty |
|---|---|---|---|
| 4.0 | 9496 | 0.74 | 0.01 |
| 4.1 | 9079 | 0.85 | 0.01 |
| 4.2 | 8428 | 0.98 | 0.01 |
| 4.3 | 7421 | 1.09 | 0.01 |
| 4.4 | 6213 | 1.19 | 0.02 |
| 4.5 | 4845 | 1.23 | 0.02 |
| 4.6 | 3583 | 1.21 | 0.02 |
| 4.7 | 2638 | 1.17 | 0.02 |
| 4.8 | 2100 | 1.23 | 0.03 |
| 4.9 | 1678 | 1.31 | 0.03 |
| 5.0 | 1221 | 1.30 | 0.04 |
| 5.1 | 859 | 1.22 | 0.04 |
| 5.2 | 625 | 1.17 | 0.05 |
| 5.3 | 462 | 1.13 | 0.05 |
| 5.4 | 350 | 1.12 | 0.06 |
| 5.5 | 265 | 1.09 | 0.07 |


4. Set `min_magnitude` in `params.yaml`, save, and rebuild. No `--forceall` this time: the changed parameter is enough. Watch which jobs run, and read the new number.

```bash
snakemake --cores 1
cat metrics.json
```

5. Commit. In the message, name the magnitude you chose and say what in the table supports it.

```bash
git add Snakefile params.yaml metrics.json
git commit -m "REPLACE THIS with the start magnitude you chose, and what in the table supports it"
```

## Step 3 (optional): declare the step the README skipped, then delete everything

**Facts**

- The README never mentions `completeness.py`. The table your decision rests on isn't part of the result, so nobody can rebuild it.
- A rule that writes `completeness.md`, listed in `rule all`, makes the table part of the workflow. Commit the file, so a reader sees the evidence without running anything.
- Every output is cattle. Delete them all, and one command brings them back byte for byte. If it doesn't, the project doesn't hold the result.

**Steps**

1. Add a rule for `completeness.py`. It reads `data/earthquakes.csv`, `params.yaml` and `completeness.py`, and writes `completeness.md`. Add `"completeness.md"` to the inputs of `rule all`.
2. Build, and commit the table with the Snakefile.

```bash
snakemake --cores 1
git add Snakefile completeness.md
git commit -m "Declare the completeness table as part of the workflow"
```

3. Delete every output and rebuild. `git status --short` at the end should print nothing: the committed files came back byte for byte.

```bash
rm -rf results .snakemake metrics.json completeness.md
snakemake --cores 1
git status --short
```

## Check everything

- The full check, for steps 1 to 3. It clones your last commit, builds from the raw data, changes a parameter, edits a script and deletes every output, and checks what your workflow does each time.
- It knows the accepted start magnitudes only as salted digests, so reading `check.py` won't give you one.
- It works in a throwaway clone, so your project is safe. It takes under a minute.

```bash
python ../check.py
```

On the starter project it fails like this:

In [6]:
# Recorded output. You don't need to run this.
run("python ../check.py")

$ python ../check.py
Note: you have uncommitted changes. This checks your last commit only.

FAIL  No Snakefile is committed. Write the rules in step 1, then git add Snakefile and commit.
FAIL  The committed min_magnitude is not a start magnitude this exercise accepts. Read the table from python completeness.py, and pick the lowest start magnitude at which b has stopped rising (step 2).
FAIL  completeness.md is not committed. Declare a rule that writes it, add it to rule all, build it and commit it (step 3).

0 of 3 parts pass.


## Summary

- A README order is a procedure. A Snakefile is a declaration: each step names what it reads and what it writes, and the tool knows what's stale.
- Declare the parameters and the code too. Otherwise a change leaves a result the tool calls up to date.
- A decision such as the start magnitude goes in `params.yaml` and the commit message, and the table behind it is an output of the workflow.
- If you can delete every output and get them back with one command, they're cattle, and the project holds the result.
- The catalog is 2.7 MB, and Git holds it. Exercise 3 does the same thing with DVC stages for data too big for Git: each stage declares what it reads and writes, and the data goes to a bucket.
- Nextflow does what Snakemake does. Snakemake wildcards run the same rules over many inputs. The idea doesn't change.

## On your own project

- Find the README, script or notebook cell that says "run these in this order". Write it as rules.
- Add every file a step reads to its inputs: the parameters, the script, the environment file.
- Delete every output and rebuild. Commit the ones a reader needs.

## About the data

- `data/earthquakes.csv`, 2.7 MB, is every earthquake of magnitude 4.0 and above in 2024, worldwide: 14,175 rows, unchanged from the [USGS ANSS Comprehensive Earthquake Catalog](https://earthquake.usgs.gov/data/comcat/) through its [event web service](https://earthquake.usgs.gov/fdsnws/event/1/), downloaded November 8, 2026.
- It's a work of the United States Government, in the public domain. USGS: "USGS-authored or produced data and information are considered to be in the U.S. Public Domain." ([Copyrights and credits](https://www.usgs.gov/information-policies-and-instructions/copyrights-and-credits))
- Cite it as: U.S. Geological Survey, Earthquake Hazards Program, 2017, Advanced National Seismic System (ANSS) Comprehensive Catalog of Earthquake Events and Products: Various, [doi:10.5066/F7MS3QZH](https://doi.org/10.5066/F7MS3QZH).
- The catalog mixes magnitude types, mostly `mb`, with `mww` for the largest events. A seismologist would treat that before trusting the b-value. The exercise is about the workflow, and the number is what this catalog gives.